# Module 2 Lab: Hands-on Data Exploration

**Practical Machine Learning Foundations**

**Purpose:** load a messy real-world-style CSV, profile it with summary statistics, find and fix its data quality problems, visualize its distributions, and check correlations, applying everything from the preceding lessons.

**Date:** 2026-08-24 | **Author:** Nick Garner

### The scenario

You are an analyst at a company whose security team exported a **web connection log** for you. Like most real exports, it is a mess: mixed types, missing values, duplicates, impossible values, and at least one column that is too good to be true.

Your job is the full exploration workflow:

| Section | Lab task |
|---|---|
| 1. Load CSV into pandas | Loading, and surviving loading errors |
| 2. Summary statistics | describe, dtypes, nunique, type conversions |
| 3. Data quality problems | Missing data, duplicates, invalid values, schema checks |
| 4. Select, filter, group | The core pandas analysis skills |
| 5. Plot distributions | Histograms, boxplots, bar charts |
| 6. Check correlations | Correlation matrix, and catching data leakage |
| 7. Enrich with joins | merge, join pitfalls, aggregation features |


## Section 0: Environment and workspace setup

**Where am I running?** Colab runs on a **temporary virtual machine** in Google's cloud. Its local filesystem is wiped when the session ends. Three ways to work with data:

1. **Upload directly**: quick but temporary, gone when the session ends
2. **Mount Google Drive**: persistent storage that survives restarts. One line, files appear at `/content/drive/MyDrive/`, and a dedicated course folder like `MyDrive/ML-Course/` is the recommended home for anything you want to keep
3. **Download from URLs**: pull public datasets straight from the internet

If you want persistence today, uncomment the Drive mount below. This lab works either way because our setup cell generates the data files locally.

**Which runtime?** Everything here is pandas work, so the default **CPU** runtime is correct. GPUs (thousands of parallel cores) shine for deep learning training, large matrix math, and image processing; TPUs are Google's custom chips optimized for TensorFlow. GPU does not speed up CSV loading, pandas wrangling, or scikit-learn models, and data transfer overhead can erase small gains.

Rule of thumb: if it runs under 30 seconds on CPU, switching is not worth it. Also remember: **Runtime > Change runtime type** restarts your session and wipes all variables, so save work to Drive first. Free tier sessions also time out after inactivity (roughly 90 minutes): Reconnect may preserve state, Restart runtime clears variables but keeps code, and the resource panel in the lower left shows your RAM and disk (free Colab gives about 12GB of RAM).  There is also a max run time of 12 hours for a notebook in the free tier.

**Security posture from the start:** we will never hardcode credentials in cells. In Colab, store secrets behind the key icon and read them with `userdata.get('KEY_NAME')`. And treat any shared notebook as executable code from an unknown source: review it before running, especially cells starting with `!`, which run arbitrary shell commands.


In [ ]:
# Optional (Colab only): mount Google Drive for persistent storage
# from google.colab import drive
# drive.mount('/content/drive')
# course_dir = '/content/drive/MyDrive/ML-Course/'

# Optional (Colab only): read a credential safely, never hardcode it
from google.colab import userdata
api_key = userdata.get('KAGGLE_KEY')


In [ ]:
# Standard imports. Documenting library versions is the minimum
# reproducibility step; pinning (e.g. !pip install scikit-learn==1.4.0)
# is the stronger option when exact behavior matters.
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("pandas", pd.__version__, "| NumPy", np.__version__)

# Fixed random seed so every run of this notebook produces identical results
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


### Project folder structure

The file organization strategy from the preceding lessons: a standard hierarchy (`data/`, `notebooks/`, `models/`, `outputs/`), with `data/raw/` separated from `data/processed/`. The original file goes in `raw/` and is **never modified**; cleaned versions go in `processed/` with descriptive, dated filenames like `connections_2026-08-24_cleaned.csv`. A file named `data_final_v2_ACTUAL_final.csv` tells you nothing. Note these are **relative paths** (`data/raw/...`, not `/home/nick/data/...`), so the notebook works on anyone's machine.


In [ ]:
for folder in ["data/raw", "data/processed", "models", "outputs"]:
    os.makedirs(folder, exist_ok=True)
print("Project folders ready:", sorted(os.listdir(".")))


In [ ]:
# SETUP CELL: generates the messy files our "security team" exported.
# In real life these arrive from upstream systems.
# (Do not worry about how it works, it exists so the lab has no reliance on external data.)
n = 4000
n_susp = 220

proto = rng.choice(["HTTP", "HTTPS", "SSH", "FTP", "DNS"], n, p=[.3,.35,.12,.08,.15])
is_susp = np.zeros(n, dtype=int); is_susp[:n_susp] = 1
rng.shuffle(is_susp)

hours = np.where(is_susp==1, rng.normal(3,2,n), rng.normal(13,5,n)).astype(int) % 24
ts = pd.to_datetime("2026-07-01") + pd.to_timedelta(rng.integers(0,14,n), unit="D") \
     + pd.to_timedelta(hours, unit="h") + pd.to_timedelta(rng.integers(0,60,n), unit="m")

bytes_sent = np.where(is_susp==1,
                      rng.lognormal(13,1,n),
                      rng.lognormal(9,1.4,n)).round(0)
bytes_str = bytes_sent.astype(int).astype(str)
bad_idx = rng.choice(n, 90, replace=False)          # garbage strings -> to_numeric coerce
bytes_str[bad_idx[:60]] = "N/A"
bytes_str[bad_idx[60:]] = "-1"                       # impossible negatives

duration = np.abs(rng.normal(45, 30, n)).round(1)
susp_idx = np.where(is_susp==1)[0]                   # MNAR: missing mostly for
hide = rng.choice(susp_idx, int(len(susp_idx)*0.85), replace=False)  # suspicious rows
duration[hide] = np.nan
noise = rng.choice(np.where(is_susp==0)[0], int(n*0.02), replace=False)
duration[noise] = np.nan                             # plus a trickle of random gaps

severity = rng.choice(["Low","Medium","High","Critical"], n, p=[.55,.28,.12,.05]).astype(object)
severity[rng.choice(n, 120, replace=False)] = None   # MCAR-ish gaps

user_agent = rng.choice(["Chrome","Firefox","Edge","curl"], n).astype(object)
user_agent[proto=="SSH"] = None                      # MAR: missing depends on protocol

legacy_notes = np.full(n, None, dtype=object)        # a mostly-dead column
keep = rng.choice(n, int(n*0.25), replace=False)
legacy_notes[keep] = "migrated"

dst_port = rng.choice([80,443,22,21,53,8080,3389], n).astype(float)
dst_port[rng.choice(n, 5, replace=False)] = 70099    # impossible port

blocked = is_susp.copy()                             # the leaky column
flip = rng.choice(n, int(n*0.03), replace=False)
blocked[flip] = 1 - blocked[flip]

df_raw = pd.DataFrame({
    "conn_id": np.arange(1, n+1),
    "timestamp": ts.astype(str),                     # dates stored as strings
    "src_ip": ["10.0."+str(a)+"."+str(b) for a,b in
               zip(rng.integers(0,20,n), rng.integers(1,255,n))],
    "dst_port": dst_port,
    "protocol": proto,
    "bytes_sent": bytes_str,                         # numbers stored as strings
    "duration_sec": duration,
    "severity": severity,
    "user_agent": user_agent,
    "encrypted": ( (proto=="HTTPS") | (proto=="SSH") ).astype(int),       # boolean stored as 0/1 int
    "legacy_notes": legacy_notes,
    "blocked_by_ids": blocked,
    "is_suspicious": is_susp,
})
df_raw = pd.concat([df_raw, df_raw.sample(60, random_state=1)])  # inject duplicates
df_raw.to_csv("data/raw/connections.csv", index=False)

# A European firewall export: semicolon-separated AND latin-1 encoded
with open("data/raw/firewall_export.csv", "w", encoding="latin-1") as f:
    f.write("device;location;région;alerts\n")
    f.write("fw-01;Zürich;Suisse;14\nfw-02;München;Bavière;9\nfw-03;Malmö;Scanie;3\n")

# A corrupted export with a malformed row (too many fields)
with open("data/raw/malformed.csv", "w") as f:
    f.write("id,host,status\n1,web-01,ok\n2,web-02,ok,EXTRA,FIELDS\n3,web-03,down\n")

print("Wrote:", os.listdir("data/raw"))

print("Go take a look at the files it created in data/raw")

---
# Section 1: Load CSV into pandas

The one-liner you will use thousands of times is `pd.read_csv(path)`. As noted in the preceding lessons, more students get stuck on file loading than on anything algorithmic, let's deliberately hit the four classic loading errors and fix each one.

### Error 1: FileNotFoundError

The most common error of all: a typo in the path, the wrong directory, or an unmounted Drive. In Colab, use the file browser in the left sidebar to copy the exact path.


In [ ]:
try:
    df = pd.read_csv("data/raw/conections.csv")   # typo: conections
except FileNotFoundError as e:
    print("FileNotFoundError:", e)
    print("Fix: check the path. Files actually present:", os.listdir("data/raw"))


### Errors 2 and 3: encoding and delimiter problems

Files exported from Excel or European systems often use `latin-1` encoding, and many exports use `;` or tab (`sep='\t'`) instead of commas. The symptoms: a `UnicodeDecodeError`, or data that loads as one giant column. A useful diagnostic in Colab is `!head data/raw/firewall_export.csv` to eyeball the raw file (remember: audit `!` shell commands in any notebook you did not write).


In [ ]:
# Symptom 1: UnicodeDecodeError on a non-UTF-8 file
try:
    fw = pd.read_csv("data/raw/firewall_export.csv")
except UnicodeDecodeError as e:
    print("UnicodeDecodeError, this file is not UTF-8. Trying encoding='latin-1'...")


In [ ]:
# Fix the encoding... and hit symptom 2: everything lands in ONE column
fw = pd.read_csv("data/raw/firewall_export.csv", encoding="latin-1")
print(fw.head(2))
print("Columns:", list(fw.columns), "<- one column means the delimiter is wrong")

In [ ]:
# Fix both: correct encoding AND the semicolon separator
fw = pd.read_csv("data/raw/firewall_export.csv", encoding="latin-1", sep=";")
fw


### Error 4: parse errors from malformed rows

A row with mismatched columns raises a `ParserError`. `on_bad_lines='skip'` skips the broken rows so you can load the rest. (Historical note from the slides: the old `error_bad_lines=False` argument was removed in pandas 2.0, `on_bad_lines` is the current syntax.)


In [ ]:
# Try to open data/raw/malformed.csv in colab, it will fail
! cat data/raw/malformed.csv

print()
try:
    bad = pd.read_csv("data/raw/malformed.csv")
except pd.errors.ParserError as e:
    print("ParserError:", str(e).strip()[:80], "...")

bad = pd.read_csv("data/raw/malformed.csv", on_bad_lines="skip")
print("\nLoaded with on_bad_lines='skip':")
print(bad)




### Loading the main dataset, and verifying it

Two more tricks before the real load. For huge files that threaten Colab's ~12GB of RAM, load a sample first with `nrows=1000` and verify your pipeline before committing to the full file. And pandas can read straight from the internet: `pd.read_csv('https://example.com/data.csv')` works with no download step (other remote options: the Kaggle API with a stored key, `!git clone` for GitHub repos, and scikit-learn's built-in datasets like `from sklearn.datasets import load_iris` for instant practice data).

**Always verify immediately after loading**: shape, first rows, and dtypes. And verify *integrity*: does the row count match what the source claimed? A corrupted download or an HTML error page saved as `.csv` still looks like a file.


In [ ]:
# Memory-safety preview: load just a sample first
sample = pd.read_csv("data/raw/connections.csv", nrows=1000)
print("Sample shape:", sample.shape)

# Now the full load
df = pd.read_csv("data/raw/connections.csv")

# Verification: shape, peek, dtypes
print("Full shape:", df.shape)
expected_rows = 4060   # what the security team told us they exported
assert df.shape[0] == expected_rows, "Row count mismatch, investigate the export!"
print("Integrity check passed: row count matches the source's claim")
df.head(3)


---
# Section 2: Summary statistics and data types

Profiling starts with three commands: `df.dtypes` (how each column is *stored*), `df.nunique()` (how many distinct values), and `df.describe()` (statistics for numeric columns). Together they reveal the gap between how data is stored and what it *means*, one of the most costly mistakes in ML.


In [ ]:
print(df.dtypes)


In [ ]:
print(df.nunique().sort_values())


In [ ]:
df.describe().round(2)

In [ ]:
print((df["dst_port"] == 80).sum(), "rows where dst_port=80")

### Reading the profile with the data type taxonomy

- **Continuous** (differences and magnitudes are meaningful, can be averaged and summed): `duration_sec` is genuinely continuous, and 100 seconds really is twice 50.
- **Categorical, nominal** (no inherent order): `protocol`, `src_ip`, `user_agent`. HTTP is not "more than" FTP, so before modeling these need **one-hot encoding** (a binary column per category), never naive integers. The slides' cautionary tale: encoding attack types as 1 through 10 led a model to predict "attack type 5" by averaging types 3 and 7. For high-cardinality columns like `src_ip`, one-hot is impractical (hundreds of new columns), so use **binary encoding** or engineer subnet/GeoIP/ASN features instead.
- **Categorical, binary**: `encrypted` and `is_suspicious`, stored as 0/1 integers. Fine as-is, just know they are categories, not quantities.
- **Ordinal** (ordered, but unequal spacing): `severity` (Low < Medium < High < Critical). **Label encoding** to ordered integers (Low=1 ... Critical=4) preserves the ranking, and is appropriate *only* for ordinal data. Tree-based models like random forests handle this fine because splits are binary; linear models assume equal spacing, so domain-informed values (Low=1, Medium=3, High=7, Critical=10) can work better there.
- **Disguises**: `dst_port` is stored as float64, but ports are identifiers, not quantities (port 443 is not "twenty times" port 22), so treat them as categorical. `conn_id` is a numeric-looking ID with zero predictive meaning. `timestamp` is a date stored as a string.

### Converting types deliberately

The conversion workflow from the preceding lessons: profile, identify mismatches, convert with error handling, verify no data loss, and document why.


In [ ]:
print("Rows before conversions:", len(df))


# 1) timestamp: string -> real datetime. Specifying format beats letting
#    pandas guess, ambiguous day/month order gets silently misread otherwise.
df["timestamp"] = pd.to_datetime(df["timestamp"], format="%Y-%m-%d %H:%M:%S")

# 2) severity: ordinal -> ordered integer codes (label encoding, ordinal only!)
severity_order = {"Low": 1, "Medium": 2, "High": 3, "Critical": 4}
df["severity_code"] = df["severity"].map(severity_order)

# 3) protocol: low-cardinality string -> category dtype (big memory win)
mem_before = df["protocol"].memory_usage(deep=True)
df["protocol"] = df["protocol"].astype("category")
mem_after = df["protocol"].memory_usage(deep=True)
print("protocol memory:", mem_before, "->", mem_after, "bytes,",
      round(100*(1-mem_after/mem_before)), "% smaller")

# Verify no data loss: same row count, dtypes now match semantics
assert len(df) == 4060
print("\nDtypes after conversion:")
print(df[["timestamp","bytes_sent","severity_code","protocol"]].dtypes)


**Memory management side note:** the category trick above is one of three memory tools for Colab's ~12GB RAM. The second is **downcasting**: `float64 -> float32` and `int64 -> int32` halve memory with no practical precision loss for most data. The third is explicitly freeing what you no longer need:


In [ ]:
# Downcasting demo
df["duration_sec"] = df["duration_sec"].astype("float32")
df["dst_port"] = pd.to_numeric(df["dst_port"], downcast="integer")

# Free the loading sample we no longer need: del plus garbage collection
del sample
gc.collect()
print("Sample released. Total DataFrame memory:",
      round(df.memory_usage(deep=True).sum() / 1024, 1), "KB")


---
# Section 3: Identify data quality problems

Real data arrives broken. We will hunt four problem families: missing values, duplicates, invalid values, and one suspicious column we will keep an eye on. First rule of cleaning: **always work on a copy, never the original**. Our raw file sits untouched in `data/raw/`, and we now branch a working copy.


In [ ]:
clean = df.copy()   # all cleaning happens on the copy


### 3.1 Missing values: detect, diagnose, then treat

Detection is two lines: `isnull().sum()` for counts, `isnull().mean() * 100` for severity. Severity drives strategy: 2% missing is a very different problem from 40% missing.


In [ ]:
missing = pd.DataFrame({
    "missing_count": clean.isnull().sum(),
    "missing_pct": (clean.isnull().mean() * 100).round(1)
})
missing[missing["missing_count"] > 0].sort_values("missing_pct", ascending=False)


### Visualizing the missingness pattern

A missingness matrix shows *where* the gaps are and whether they co-occur. The `missingno` library's `msno.matrix(df)` is the standard tool; the same picture can be drawn with plain matplotlib:


In [ ]:
cols_with_gaps = ["legacy_notes", "duration_sec", "severity",
                  "user_agent", "bytes_sent"]
plt.figure(figsize=(9, 4))
plt.imshow(clean[cols_with_gaps].isnull().T, aspect="auto",
           cmap="gray_r", interpolation="nearest")
plt.yticks(range(len(cols_with_gaps)), cols_with_gaps)
plt.xlabel("Row number")
plt.title("Missingness Matrix (dark = missing)")
plt.show()

### Diagnose: WHY is each column missing? (MCAR, MAR, MNAR)

The critical question from the preceding lessons. The mechanism determines the fix:

- **MCAR (Missing Completely at Random)**: truly random gaps, like random sensor failures. Safe for simple methods. Our `severity` gaps look like this.
- **MAR (Missing at Random)**: missingness depends on *other observed* columns. Check `user_agent`: is it missing everywhere, or only for certain protocols?
- **MNAR (Missing Not at Random)**: the missingness itself carries information. Check `duration_sec` against our label.


In [ ]:
# MAR check: user_agent missing rate by protocol
print("user_agent missing rate by protocol:")
print(clean.groupby("protocol", observed=True)["user_agent"]
      .apply(lambda s: round(s.isnull().mean()*100, 1)))

# MNAR check: duration_sec missing rate by suspicion label
print("\nduration_sec missing rate by is_suspicious:")
print(clean.groupby("is_suspicious")["duration_sec"]
      .apply(lambda s: round(s.isnull().mean()*100, 1)))


Diagnosis complete. `user_agent` is missing *only* for SSH connections (SSH clients do not send browser agents): **MAR**, explained by another column. And `duration_sec` is missing for the vast majority of suspicious connections but only a sliver of legitimate ones: **MNAR**, the missingness itself carries information, and in a security context that is a screaming signal, not noise. The security framing we covered earlier: missing logs may indicate tampering, incomplete flow records can signal evasion attacks, and vanished re-authentication events can mean session hijacking. The absence of data is often the most important data you have.

**The technique that captures this signal:** create a binary indicator column *before* imputing, so the model receives both the filled value and the fact that it was originally missing. The slides call this one of the most underused tricks in security data science.


In [ ]:
# 1) Preserve the MNAR signal as a feature BEFORE touching the gaps
clean["duration_was_missing"] = clean["duration_sec"].isnull().astype(int)

# 2) Now impute duration with the MEDIAN: it resists outliers, unlike the
#    mean, which gets dragged by our skewed distribution
clean["duration_sec"] = clean["duration_sec"].fillna(
    clean["duration_sec"].median())

# 3) Categorical gaps get the MODE (most common value)
clean["severity"] = clean["severity"].fillna(clean["severity"].mode()[0])
clean["severity_code"] = clean["severity"].map(
    {"Low":1, "Medium":2, "High":3, "Critical":4})

# 4) user_agent: the gap is MAR and meaningful, fill with an explicit label
clean["user_agent"] = clean["user_agent"].fillna("none_ssh")

# 5) bytes_sent: only ~1.5% missing and effectively random, so dropping those
#    rows is acceptable (under 5% and MCAR is the threshold for dropna).
#    Use subset= so we only drop rows missing THIS column, far less destructive
#    than a bare dropna().
clean = clean.dropna(subset=["bytes_sent"])

# 6) legacy_notes is ~74% missing: a column that empty is not worth keeping.
#    Rule of thumb: drop columns above ~50% missing.
clean = clean.drop(columns=["legacy_notes"])

print("Remaining missing values per column should be zero:")
print(clean.isnull().sum().sum(), "missing cells total")


**Other strategies in the toolkit** (right tool depends on column, missingness type, and domain):

- **Forward / backward fill** for time series where state persists: `df.ffill()` carries the last known value forward (the old `fillna(method='ffill')` syntax was removed in pandas 2.2)
- **Time interpolation** for gaps in a numeric time series: `s.interpolate(method='time')` draws a time-weighted line between known neighbors
- **KNN imputation**: `KNNImputer(n_neighbors=5)` fills gaps using the most similar complete records, capturing relationships simple fills miss
- **Iterative imputation (MICE)**: `IterativeImputer()` models each feature as a function of all the others, cycling until estimates converge

A quick look at the first three:


In [ ]:
# Forward fill and time interpolation on a tiny sensor series with gaps
sensor = pd.Series([10.0, np.nan, np.nan, 16.0, np.nan, 22.0],
                   index=pd.date_range("2026-08-24 00:00", periods=6, freq="h"),
                   name="temp_C")
demo = pd.DataFrame({"raw": sensor,
                     "ffill": sensor.ffill(),
                     "time_interp": sensor.interpolate(method="time")})
print(demo)


In [ ]:
# KNN imputation on our numeric columns (small demo on a sample)
from sklearn.impute import KNNImputer
num_cols = ["bytes_sent", "duration_sec", "severity_code"]
demo_gaps = df[num_cols].head(200).copy()          # from pre-cleaning df, has gaps
imputer = KNNImputer(n_neighbors=5)
filled = pd.DataFrame(imputer.fit_transform(demo_gaps), columns=num_cols)
print("Gaps before KNN:", demo_gaps.isnull().sum().sum(),
      "| after:", filled.isnull().sum().sum())
# MICE, for reference:
# from sklearn.experimental import enable_iterative_imputer
# from sklearn.impute import IterativeImputer


**Why all this care?** Most scikit-learn algorithms crash outright on NaN, careless dropping wastes training data, and bad imputation manufactures patterns that do not exist. Different strategies per column, chosen by mechanism, is the professional standard. Document your assumptions as you go (as this notebook's markdown is doing): record *why* each column was handled its way, not just what you did.

### 3.2 Duplicate records


In [ ]:
dup_count = clean.duplicated().sum()
print("Fully duplicated rows:", dup_count)
clean = clean.drop_duplicates()
print("After drop_duplicates:", clean.shape)


Sixty duplicated rows, exactly the injection our "export" contained. Duplicates are dangerous beyond wasted space: if the same record lands in both training and test sets later, that is **train-test contamination**, a form of data leakage (more in Section 6).

### 3.3 Invalid values and schema validation

As we learned in the preceding lessons, treat schema validation as **input validation for your ML pipeline**, the same zero-trust principle as any security system. Upstream sources change without warning (columns renamed, types changed, new categories, ranges shifting), and a model trained on clean data breaks *silently* on dirty data. The slides' real example: a database migration changed integers to strings, the pipeline coerced them to NaN and filled with zeros, and the model predicted garbage for weeks.

Hunt for impossible values, then encode the expectations as assertions:


In [ ]:
# Impossible values: negative byte counts, ports outside 0-65535
print("Negative bytes_sent:", (clean["bytes_sent"] < 0).sum())
print("Ports above 65535: ", (clean["dst_port"] > 65535).sum())

# These rows are corrupt beyond repair, remove them
clean = clean[(clean["bytes_sent"] >= 0) & (clean["dst_port"] <= 65535)]
print("Shape after removing invalid rows:", clean.shape)


In [ ]:
# A reusable validation function: our data contract, enforced.
# pandas assertions like these fit lab work and simple projects; production
# pipelines graduate to Great Expectations or pandera for the same job
# with reporting built in.
def validate_schema(d):
    expected_cols = {"conn_id", "timestamp", "src_ip", "dst_port", "protocol",
                     "bytes_sent", "duration_sec", "severity", "user_agent",
                     "encrypted", "blocked_by_ids", "is_suspicious"}
    valid_severities = {"Low", "Medium", "High", "Critical"}

    assert expected_cols.issubset(d.columns), "Missing expected columns!"
    assert d["conn_id"].notna().all(),        "Null IDs found!"
    assert d["conn_id"].is_unique,            "Duplicate IDs found!"
    assert d["bytes_sent"].between(0, 1e12).all(),  "bytes_sent out of range!"
    assert d["dst_port"].between(0, 65535).all(),   "Invalid port numbers!"
    assert d["severity"].isin(valid_severities).all(), "Unknown severity value!"
    assert d["is_suspicious"].isin([0, 1]).all(),   "Label must be binary!"
    return "Schema validation PASSED: " + str(d.shape[0]) + " rows"

print(validate_schema(clean))


Run a function like this at the start of every pipeline execution: minutes to write, hours of debugging saved. Beyond the lab, three more practices belong in your mental model:

- **Schema evolution**: adding a column is an *additive* change (safe); renaming, removing, or retyping is a *breaking* change requiring pipeline updates and usually retraining. Version schemas alongside models (Model v2.1 records Schema v2.1), use semantic versioning (major for breaking, minor for additive), and keep a changelog.
- **Automated drift monitoring**: production systems watch incoming data continuously. The **KS test** compares continuous feature distributions, the **chi-squared test** compares categorical frequencies, and the **Population Stability Index (PSI)** summarizes overall shift (under 0.1 fine, 0.1 to 0.2 watch, over 0.2 investigate). Manual checks do not scale.
- **Security**: weak validation invites **data poisoning** via subtly malformed inputs, **evasion** through unvalidated assumptions (negative values or extremes pushing inputs into feature-space regions the model never saw, the ML cousin of a buffer overflow), and even **denial of service** if violations crash the pipeline instead of being logged. Validate at ingestion, preprocessing, and inference (defense in depth), handle violations gracefully, and log every one with timestamp, source, and violation type for forensics. Security models deserve tighter alert thresholds than recommenders: a sudden shift might be an adversary probing, not benign drift.

Save the cleaned result with a descriptive, dated name, raw file untouched:


In [ ]:
out_path = "data/processed/connections_2026-08-24_cleaned.csv"
clean.to_csv(out_path, index=False)
print("Saved:", out_path, "| shape:", clean.shape)


---
# Section 4: Select, filter, group

The cleaned data is ready for questions.

### Selecting: columns, rows by label, rows by position

Single brackets return a Series, double brackets a DataFrame. For rows: `.loc` is **label-based** (looking up a contact by name), `.iloc` is **position-based** (grabbing by place in line).


In [ ]:
print(type(clean["protocol"]), "| single brackets -> Series")
print(type(clean[["protocol", "bytes_sent"]]), "| double brackets -> DataFrame")

# Index by conn_id to make .loc meaningful
byid = clean.set_index("conn_id")
print("\n.loc[3] (label lookup, the row whose conn_id IS 3):")
print(byid.loc[3, ["protocol", "bytes_sent", "is_suspicious"]])
print("\n.iloc[0:3] (position slice, first three rows like a Python list):")
print(byid.iloc[0:3][["protocol", "bytes_sent"]])


### Filtering: boolean masks and the whole toolkit

Combine conditions with `&` (and), `|` (or), `~` (not), each wrapped in parentheses (skip the parentheses and you get cryptic operator-precedence errors). Then the specialist filters: `.isin()` for known lists, `.str.contains()` / `.str.startswith()` / `.str.match()` for text and regex, `.between()` for ranges, `.isna()` / `.notna()` for gaps, `.query()` for SQL-flavored strings (with `@variable` references), and the `.dt` accessor for time components.


In [ ]:
# Boolean mask with & and parentheses: big unencrypted transfers
big_unenc = clean[(clean["bytes_sent"] > 1_000_000) & (clean["encrypted"] == 0)]
print("Large unencrypted transfers:", len(big_unenc))

# .isin(): match against a known-bad list (blocklist pattern)
watchlist = ["10.0.3.14", "10.0.7.77", "10.0.12.203"]
print("Connections from watchlisted IPs:", clean["src_ip"].isin(watchlist).sum())

# ~ negation: everything NOT in an allowlist (exclusion filtering)
allowed_ports = [80, 443, 53]
offlist = clean[~clean["dst_port"].isin(allowed_ports)]
print("Connections to non-standard ports:", len(offlist))

# String filters: .str.startswith and .str.contains for log-style matching
subnet3 = clean[clean["src_ip"].str.startswith("10.0.3.")]
print("Connections from the 10.0.3.x subnet:", len(subnet3))
print("User agents containing 'curl':",
      clean["user_agent"].str.contains("curl").sum())

# .between(): a range check in one readable call
mid = clean[clean["duration_sec"].between(30, 60)]
print("Durations between 30 and 60 sec:", len(mid))

# .dt accessor: slice by time components, e.g. after-hours events
late = clean[clean["timestamp"].dt.hour.between(0, 5)]
print("Connections between midnight and 5 AM:", len(late))

# .query(): SQL-like string syntax, with @ for outside variables
threshold = 3
crit = clean.query("severity_code >= @threshold and encrypted == 0")
print("Unencrypted high/critical severity:", len(crit))


(`.str.match()` takes a full regex for pattern-shaped hunting, like hostnames following known C2 naming conventions: `df[df['url'].str.match(r'^https://.*\.gov$')]`.)

### Grouping: split, apply, combine

`groupby` splits rows into groups, applies a function to each, and combines the results, identical to SQL's GROUP BY. `.agg()` extends it: multiple statistics at once, named output columns, multiple group keys.


In [ ]:
# Basic: mean bytes per protocol
print(clean.groupby("protocol", observed=True)["bytes_sent"].mean().round(0))

# Multiple aggregations at once
print("\n", clean.groupby("protocol", observed=True)["duration_sec"]
      .agg(["mean", "count", "std"]).round(1))


In [ ]:
# Named aggregations + multiple group keys: suspicion rate by protocol and encryption
summary = (clean
    .groupby(["protocol", "encrypted"], observed=True)
    .agg(connections=("conn_id", "count"),
         avg_bytes=("bytes_sent", "mean"),
         suspicion_rate=("is_suspicious", "mean"))
    .round({"avg_bytes": 0, "suspicion_rate": 3}))
summary


### Chaining: the fluent pipeline

Real pandas fluency is chaining steps into one readable pipeline with no intermediate variables, wrapped in parentheses for multi-line clarity. The security example from the preceding lessons, realized: find the source IPs behind the most suspicious traffic.


In [ ]:
top_offenders = (
    clean[clean["is_suspicious"] == 1]        # filter to suspicious rows
    [["src_ip", "bytes_sent"]]                # select needed columns early
    .groupby("src_ip")                        # group by source
    .agg(events=("bytes_sent", "count"),      # aggregate
         total_bytes=("bytes_sent", "sum"))
    .sort_values("events", ascending=False)   # sort
    .head(5)                                  # display top results
)
top_offenders


### Performance: vectorize, always

At a million rows the difference between vectorized operations and Python loops is 2 seconds versus 20 minutes: vectorized operations run in compiled C, while loops and `apply()` run in the Python interpreter. Selecting only needed columns early (as the chain above does) also shrinks every downstream step. Measure with the `%%time` cell magic:


In [ ]:
%%time
# SLOW: apply() with a lambda runs Python code per row
_ = clean["bytes_sent"].apply(lambda x: x * 2)


In [ ]:
%%time
# FAST: the vectorized version, same result, compiled C underneath
_ = clean["bytes_sent"] * 2


---
# Section 5: Plot distributions

The rule for continuous features: always visualize before modeling. Distributions matter (normal, skewed, bimodal), and heavily skewed features often need transformation. Three primary views: the histogram, the boxplot, and the bar chart.


In [ ]:
clean.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: raw histogram of bytes_sent
axes[0].hist(clean["bytes_sent"], bins=60, color="steelblue", edgecolor="white")
axes[0].set_title("bytes_sent: Raw Scale (heavily skewed)")
axes[0].set_xlabel("Bytes sent")
axes[0].set_ylabel("Connections")

# Right: same data on a log scale, structure becomes visible
axes[1].hist(np.log10(clean["bytes_sent"] + 1), bins=60,
             color="steelblue", edgecolor="white")
axes[1].set_title("bytes_sent: Log10 Scale")
axes[1].set_xlabel("log10(bytes sent)")

plt.tight_layout()
plt.show()


The raw histogram is a wall against the left axis: extreme skew.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot: duration by suspicion label, medians, spread, and outliers at a glance
clean.boxplot(column="duration_sec", by="is_suspicious", ax=axes[0])
axes[0].set_title("Duration by Label")
axes[0].set_xlabel("is_suspicious")
axes[0].set_ylabel("Duration (sec)")

# Bar chart: ordinal severity distribution, in its meaningful order
order = ["Low", "Medium", "High", "Critical"]
clean["severity"].value_counts().reindex(order).plot(
    kind="bar", ax=axes[1], color=["#4c9be8", "#f2b705", "#e8743b", "#c02f1d"])
axes[1].set_title("Connections by Severity (ordinal)")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=0)

plt.suptitle("")
plt.tight_layout()
plt.show()


The boxplot's suspicious group sits at a single flat value: that is our median imputation showing itself, visible proof that imputation choices leave fingerprints in the data (and why we kept the `duration_was_missing` indicator). The bar chart respects the severity levels' natural order, which is what makes ordinal data ordinal.




# Wrap-up

This lab covered the mechanics: load, summarize, clean, plot, correlate.
The remaining notebook practices are what make the work durable:

**Reproducibility.** This notebook pinned nothing but printed versions, set `RANDOM_STATE = 42` everywhere randomness appears, used relative paths only, and hardcoded zero credentials.

**Version control.** Milestone filenames with dates (`connections_2026-08-24_cleaned.csv`) are the minimum. For Git, clear all outputs before committing (outputs bloat diffs and can contain sensitive data previews or internal paths), and the `nbstripout` tool strips them automatically on every commit. Colab's **File > Revision history** covers quick rollbacks but is not a substitute. And never commit API keys, even to private repos: keys persist in Git history after deletion.

**Documentation.** Notice this notebook documented *why*, not just what: why the median (skew), why an indicator column (MNAR), why the drop (leakage). Record what did not work too, and keep inline comments for non-obvious decisions only.

**Anti-patterns we avoided:** giant hundred-line cells, mystery state that depends on hidden execution order, copy-pasted notebook variants instead of reusable functions (our `validate_schema` is the model), zero error handling (our try/except blocks and asserts are the fix), and ignored warnings, since today's deprecation warning is next release's breaking error, as the `error_bad_lines` and `fillna(method=)` removals proved.

**Beyond Colab:** these exact skills (choosing compute, managing memory, organizing projects, writing reproducible notebooks) transfer directly to **AWS SageMaker** (S3 instead of Drive), **Azure ML**, and **GCP Vertex AI**, Colab's enterprise sibling. You are learning a workflow, not a tool.



---
# Try it yourself (optional)

**Exercise 1.** Using one chained pipeline: among **encrypted** connections, find the 3 protocols with the highest average `bytes_sent`. (Filter, group, aggregate, sort, head.)

**Exercise 2.** The team suspects weekend activity differs. Using the `.dt` accessor (`.dt.dayofweek`, where 5 and 6 are Saturday and Sunday), compare the suspicion rate (`is_suspicious` mean) on weekends vs. weekdays.



In [ ]:
# Exercise workspace





---
## Exercise solutions


In [ ]:
# Exercise 1: top 3 protocols by average bytes among encrypted connections
(clean[clean["encrypted"] == 1]
    .groupby("protocol", observed=True)["bytes_sent"]
    .mean()
    .sort_values(ascending=False)
    .head(3)
    .round(0))


In [ ]:
# Exercise 2: weekend vs. weekday suspicion rate
clean["is_weekend"] = clean["timestamp"].dt.dayofweek.isin([5, 6])
print(clean.groupby("is_weekend")["is_suspicious"].mean().round(4))
